In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import gradio as gr
from pathlib import Path
from tqdm import tqdm
import glob

In [2]:
os.chdir('/kaggle/input')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'GPU Count: {torch.cuda.device_count()}')

Using device: cuda
GPU Count: 2


In [3]:
class PairedImageDataset(Dataset):
    def __init__(self, *dirs, img_size=256):
        # Accept pairs of (input_dir, target_dir) as positional arguments
        if len(dirs) % 2 != 0:
            raise ValueError("Please provide directory pairs: (input_dir, target_dir, input_dir, target_dir, ...)")
        
        self.input_paths = []
        self.target_paths = []
        
        # Process each input/target directory pair
        for i in range(0, len(dirs), 2):
            input_dir = dirs[i]
            target_dir = dirs[i + 1]
            self.input_paths.extend(sorted(glob.glob(f'{input_dir}/*.jpg')) + sorted(glob.glob(f'{input_dir}/*.png')))
            self.target_paths.extend(sorted(glob.glob(f'{target_dir}/*.jpg')) + sorted(glob.glob(f'{target_dir}/*.png')))
        
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
    
    def __len__(self):
        return min(len(self.input_paths), len(self.target_paths))
    
    def __getitem__(self, idx):
        input_img = Image.open(self.input_paths[idx]).convert('RGB')
        target_img = Image.open(self.target_paths[idx]).convert('RGB')
        return self.transform(input_img), self.transform(target_img)

In [4]:
def conv_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1, activation=True):
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)]
    if activation:
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

def deconv_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1, dropout=0.0):
    layers = [nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)]
    layers.append(nn.BatchNorm2d(out_channels))
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

In [5]:
class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.enc1 = conv_block(in_channels, 64, activation=False)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)
        self.enc4 = conv_block(256, 512)
        self.enc5 = conv_block(512, 512)
        self.enc6 = conv_block(512, 512)
        self.enc7 = conv_block(512, 512)
        self.enc8 = conv_block(512, 512, activation=False)
        
        self.dec8 = deconv_block(512, 512, dropout=0.5)
        self.dec7 = deconv_block(1024, 512, dropout=0.5)
        self.dec6 = deconv_block(1024, 512, dropout=0.5)
        self.dec5 = deconv_block(1024, 512)
        self.dec4 = deconv_block(1024, 256)
        self.dec3 = deconv_block(512, 128)
        self.dec2 = deconv_block(256, 64)
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),
            nn.Tanh()
        )
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        e6 = self.enc6(e5)
        e7 = self.enc7(e6)
        e8 = self.enc8(e7)
        
        d8 = self.dec8(e8)
        d7 = self.dec7(torch.cat([d8, e7], 1))
        d6 = self.dec6(torch.cat([d7, e6], 1))
        d5 = self.dec5(torch.cat([d6, e5], 1))
        d4 = self.dec4(torch.cat([d5, e4], 1))
        d3 = self.dec3(torch.cat([d4, e3], 1))
        d2 = self.dec2(torch.cat([d3, e2], 1))
        d1 = self.dec1(torch.cat([d2, e1], 1))
        return d1

In [6]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        self.conv1 = conv_block(in_channels, 64, activation=False)
        self.conv2 = conv_block(64, 128)
        self.conv3 = conv_block(128, 256)
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.conv5 = nn.Conv2d(512, 1, 4, 1, 1)
    
    def forward(self, x, y):
        x = torch.cat([x, y], 1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        return x

In [7]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)
if torch.cuda.device_count() > 1:
    generator = nn.DataParallel(generator)
    discriminator = nn.DataParallel(discriminator)

opt_g = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999), weight_decay=1e-5)
opt_d = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999), weight_decay=1e-5)
scheduler_g = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_g, T_0=10, T_mult=1, eta_min=1e-5)
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_d, T_0=10, T_mult=1, eta_min=1e-5)
scaler = GradScaler('cuda')
criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()

In [8]:
# Load and analyze ALL available data from dataset folders
import os

# Optimized dataset pairs - using ALL available subdirectories and variations
dataset_pairs = [
    # CUHK Face-Sketch Database - original_sketch with photo (main pairing)
    ('/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/original_sketch',
     '/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photo'),
    
    # CUHK Face-Sketch Database - photos with sketches (alternative pairing)
    ('/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photos',
     '/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/sketches'),
    
    # CUHK Face-Sketch Database - cropped_sketch with photo
    ('/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/cropped_sketch',
     '/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photo'),
    
    # CUHK Face-Sketch Database - sketch with photos
    ('/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/sketch',
     '/kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photos'),
    
    # Anime Sketch Colorization - using train split
    ('/kaggle/input/datasets/ktaebum/anime-sketch-colorization-pair/data/train',
     '/kaggle/input/datasets/ktaebum/anime-sketch-colorization-pair/data/train'),
    
    # Anime Sketch Colorization - using val split (additional data)
    ('/kaggle/input/datasets/ktaebum/anime-sketch-colorization-pair/data/val',
     '/kaggle/input/datasets/ktaebum/anime-sketch-colorization-pair/data/val'),
]

# Enhanced dataset class with comprehensive file matching
class ComprehensivePairedImageDataset(Dataset):
    def __init__(self, dataset_pairs, img_size=256):
        self.input_paths = []
        self.target_paths = []
        self.dataset_sources = []  # Track source for each pair
        self.img_size = img_size
        
        print("\n" + "="*80)
        print("COMPREHENSIVE DATASET LOADING AND ANALYSIS")
        print("="*80)
        print("\nAnalyzing all available directories and subdirectories...\n")
        
        # Process each dataset pair
        for pair_idx, (input_dir, target_dir) in enumerate(dataset_pairs):
            print(f"[Dataset Pair {pair_idx + 1}/6]")
            print(f"  Input:  {input_dir}")
            print(f"  Target: {target_dir}")
            
            # Collect all image files recursively
            input_files = self._get_all_images(input_dir)
            target_files = self._get_all_images(target_dir)
            
            print(f"  ✓ Found {len(input_files):4d} input images")
            print(f"  ✓ Found {len(target_files):4d} target images")
            
            # For paired datasets, use min length to ensure proper pairing
            num_pairs = min(len(input_files), len(target_files))
            
            if num_pairs > 0:
                # Add all paired files
                for i in range(num_pairs):
                    self.input_paths.append(input_files[i])
                    self.target_paths.append(target_files[i])
                    self.dataset_sources.append(pair_idx)
                
                print(f"  ✓ Created {num_pairs:4d} paired samples")
            else:
                print(f"  ✗ Skipped - No valid image pairs")
            
            print()
        
        # Setup image transformations
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        
        print("="*80)
        print("FINAL DATASET SUMMARY")
        print("="*80)
        print(f"Total image pairs loaded:     {len(self.input_paths):6d}")
        print(f"Image resolution:             {img_size}x{img_size}")
        print(f"Number of dataset sources:    {len(set(self.dataset_sources)):6d}")
        print(f"Unique datasets:              CUHK (4 variations) + Anime (2 splits)")
        print("="*80 + "\n")
    
    def _get_all_images(self, directory):
        """Recursively get all images from a directory"""
        if not os.path.exists(directory):
            return []
        
        # Get all jpg and png files recursively
        jpg_files = sorted(glob.glob(f'{directory}/**/*.jpg', recursive=True))
        jpeg_files = sorted(glob.glob(f'{directory}/**/*.jpeg', recursive=True))
        png_files = sorted(glob.glob(f'{directory}/**/*.png', recursive=True))
        
        all_files = jpg_files + jpeg_files + png_files
        return all_files
    
    def __len__(self):
        return len(self.input_paths)
    
    def __getitem__(self, idx):
        try:
            # Load images
            input_img = Image.open(self.input_paths[idx]).convert('RGB')
            target_img = Image.open(self.target_paths[idx]).convert('RGB')
            
            # Apply transformations
            input_tensor = self.transform(input_img)
            target_tensor = self.transform(target_img)
            
            return input_tensor, target_tensor
            
        except Exception as e:
            print(f"Error loading image pair {idx}: {e}")
            print(f"  Input: {self.input_paths[idx]}")
            print(f"  Target: {self.target_paths[idx]}")
            
            # Return next valid pair if current fails
            idx = (idx + 1) % len(self.input_paths)
            input_img = Image.open(self.input_paths[idx]).convert('RGB')
            target_img = Image.open(self.target_paths[idx]).convert('RGB')
            
            return self.transform(input_img), self.transform(target_img)

# Create the comprehensive dataset
dataset = ComprehensivePairedImageDataset(dataset_pairs, img_size=256)

# Create dataloaders with optimized settings
dataloader = DataLoader(
    dataset, 
    batch_size=24, 
    shuffle=True, 
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    dataset, 
    batch_size=24, 
    shuffle=False, 
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

# Print detailed dataset statistics
print("="*80)
print("TRAINING CONFIGURATION")
print("="*80)
print(f"Total training samples:        {len(dataset):6d}")
print(f"Batch size:                    {24:6d}")
print(f"Batches per epoch:             {len(dataloader):6d}")
print(f"Total epochs:                  {50:6d}")
print(f"Total training iterations:     {len(dataloader) * 50:7,d}")
print(f"Estimated model parameter updates: {len(dataloader) * 50:7,d}")
print("\nDataset Sources:")
print("  • CUHK original_sketch → photo")
print("  • CUHK photos → sketches")
print("  • CUHK cropped_sketch → photo")
print("  • CUHK sketch → photos")
print("  • Anime train split")
print("  • Anime val split (additional)")
print("="*80 + "\n")


COMPREHENSIVE DATASET LOADING AND ANALYSIS

Analyzing all available directories and subdirectories...

[Dataset Pair 1/6]
  Input:  /kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/original_sketch
  Target: /kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photo
  ✓ Found 1194 input images
  ✓ Found  182 target images
  ✓ Created  182 paired samples

[Dataset Pair 2/6]
  Input:  /kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photos
  Target: /kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/sketches
  ✓ Found  188 input images
  ✓ Found  188 target images
  ✓ Created  188 paired samples

[Dataset Pair 3/6]
  Input:  /kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/cropped_sketch
  Target: /kaggle/input/datasets/arbazkhan971/cuhk-face-sketch-database-cufs/photo
  ✓ Found 1194 input images
  ✓ Found  182 target images
  ✓ Created  182 paired samples

[Dataset Pair 4/6]
  Input:  /kaggle/input/datas

In [9]:
def train_step(real_sketch, real_photo):
    real_sketch = real_sketch.to(device)
    real_photo = real_photo.to(device)
    
    with autocast(device_type='cuda'):
        fake_photo = generator(real_sketch)
        fake_pred = discriminator(real_sketch, fake_photo.detach())
        real_pred = discriminator(real_sketch, real_photo)
        
        loss_d_fake = criterion_gan(fake_pred, torch.zeros_like(fake_pred))
        loss_d_real = criterion_gan(real_pred, torch.ones_like(real_pred))
        loss_d = (loss_d_fake + loss_d_real) * 0.5
    
    opt_d.zero_grad()
    scaler.scale(loss_d).backward()
    scaler.step(opt_d)
    
    with autocast(device_type='cuda'):
        fake_photo = generator(real_sketch)
        fake_pred = discriminator(real_sketch, fake_photo)
        loss_g_gan = criterion_gan(fake_pred, torch.ones_like(fake_pred))
        loss_g_l1 = criterion_l1(fake_photo, real_photo) * 100
        loss_g = loss_g_gan + loss_g_l1
    
    opt_g.zero_grad()
    scaler.scale(loss_g).backward()
    scaler.step(opt_g)
    scaler.update()
    
    return loss_d.item(), loss_g.item()

In [10]:
def validate(val_dataloader):
    generator.eval()
    val_loss = 0
    with torch.no_grad():
        for real_sketch, real_photo in val_dataloader:
            real_sketch = real_sketch.to(device)
            real_photo = real_photo.to(device)
            fake_photo = generator(real_sketch)
            loss = criterion_l1(fake_photo, real_photo)
            val_loss += loss.item()
    return val_loss / len(val_dataloader)

In [ ]:
epochs = 50
checkpoint_dir = '/kaggle/working/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
best_val_loss = float('inf')

for epoch in range(epochs):
    generator.train()
    discriminator.train()
    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}')
    for sketches, photos in pbar:
        loss_d, loss_g = train_step(sketches, photos)
        pbar.set_postfix({'D_loss': f'{loss_d:.4f}', 'G_loss': f'{loss_g:.4f}'})
    
    scheduler_g.step()
    scheduler_d.step()
    
    if (epoch + 1) % 3 == 0:
        val_loss = validate(val_loader)
        print(f'Epoch {epoch+1} - Validation L1 Loss: {val_loss:.4f}')
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'generator': generator.state_dict(),
                'discriminator': discriminator.state_dict(),
                'epoch': epoch,
                'val_loss': val_loss
            }, f'{checkpoint_dir}/best_model.pt')
    
    if (epoch + 1) % 10 == 0:
        torch.save({
            'generator': generator.state_dict(),
            'discriminator': discriminator.state_dict(),
            'epoch': epoch
        }, f'{checkpoint_dir}/checkpoint_epoch_{epoch+1}.pt')

In [ ]:
def denormalize(img):
    return (img * 0.5 + 0.5).clamp(0, 1)

def visualize_results(num_samples=8):
    generator.eval()
    with torch.no_grad():
        sketches, photos = next(iter(DataLoader(dataset, batch_size=num_samples, shuffle=True)))
        sketches = sketches.to(device)
        generated = generator(sketches)
        
        fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
        for i in range(num_samples):
            sk = denormalize(sketches[i].cpu()).permute(1, 2, 0).numpy()
            gen = denormalize(generated[i].cpu()).permute(1, 2, 0).numpy()
            ph = denormalize(photos[i]).permute(1, 2, 0).numpy()
            
            axes[i, 0].imshow(sk)
            axes[i, 0].set_title('Input Sketch')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(gen)
            axes[i, 1].set_title('Generated')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(ph)
            axes[i, 2].set_title('Ground Truth')
            axes[i, 2].axis('off')
        
        plt.tight_layout()
        plt.savefig('/kaggle/working/results.png', dpi=100, bbox_inches='tight')
        plt.show()

visualize_results(10)

In [ ]:
checkpoint = torch.load(f'{checkpoint_dir}/best_model.pt')
generator.load_state_dict(checkpoint['generator'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1} with val_loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Save all models and important files to Kaggle output directory
import shutil
import json
from datetime import datetime

output_dir = '/kaggle/working'
os.makedirs(output_dir, exist_ok=True)

print("Saving all model files and data...")
print("="*60)

# 1. Save best model
best_model_path = os.path.join(checkpoint_dir, 'best_model.pt')
output_best_model = os.path.join(output_dir, 'best_model.pt')
if os.path.exists(best_model_path):
    shutil.copy(best_model_path, output_best_model)
    print(f"✓ Saved best model: {output_best_model}")

# 2. Save final checkpoint
final_checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_50.pt')
output_final = os.path.join(output_dir, 'final_checkpoint_epoch_50.pt')
if os.path.exists(final_checkpoint_path):
    shutil.copy(final_checkpoint_path, output_final)
    print(f"✓ Saved final checkpoint: {output_final}")

# 3. Save only generator model (for inference)
generator_only_path = os.path.join(output_dir, 'generator_only.pt')
torch.save(generator.state_dict(), generator_only_path)
print(f"✓ Saved generator model (inference only): {generator_only_path}")

# 4. Save model architecture as text
architecture_path = os.path.join(output_dir, 'model_architecture.txt')
with open(architecture_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("PIX2PIX MODEL ARCHITECTURE\n")
    f.write("="*60 + "\n\n")
    f.write("GENERATOR:\n")
    f.write(str(generator) + "\n\n")
    f.write("DISCRIMINATOR:\n")
    f.write(str(discriminator) + "\n")
print(f"✓ Saved model architecture: {architecture_path}")

# 5. Save training configuration
config_path = os.path.join(output_dir, 'training_config.json')
config = {
    'epochs': 50,
    'batch_size': 24,
    'learning_rate_g': 0.0002,
    'learning_rate_d': 0.0002,
    'betas': [0.5, 0.999],
    'image_size': 256,
    'dataset_size': len(dataset),
    'batches_per_epoch': len(dataloader),
    'total_training_steps': len(dataloader) * 50,
    'generator_params': sum(p.numel() for p in generator.parameters()),
    'discriminator_params': sum(p.numel() for p in discriminator.parameters()),
    'saved_at': datetime.now().isoformat(),
    'device': str(device),
    'gpu_count': torch.cuda.device_count()
}
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)
print(f"✓ Saved training config: {config_path}")

# 6. Save copy of all checkpoints
checkpoint_output_dir = os.path.join(output_dir, 'checkpoints')
if os.path.exists(checkpoint_dir):
    if os.path.exists(checkpoint_output_dir):
        shutil.rmtree(checkpoint_output_dir)
    shutil.copytree(checkpoint_dir, checkpoint_output_dir)
    print(f"✓ Copied all checkpoints to: {checkpoint_output_dir}")

# 7. Save results visualization if it exists
if os.path.exists('/kaggle/working/results.png'):
    shutil.copy('/kaggle/working/results.png', os.path.join(output_dir, 'results_visualization.png'))
    print(f"✓ Saved results visualization: {os.path.join(output_dir, 'results_visualization.png')}")

# 8. Save a README with usage instructions
readme_path = os.path.join(output_dir, 'README.md')
with open(readme_path, 'w') as f:
    f.write("""# Pix2Pix Sketch-to-Image Translation Model

## Overview
This is a trained Pix2Pix GAN model for converting sketch/grayscale images to colored/realistic images.

## Model Files

### For Training
- **best_model.pt**: Best performing model (both generator & discriminator)
- **final_checkpoint_epoch_50.pt**: Final model after 50 epochs
- **checkpoints/**: Directory containing all epoch checkpoints

### For Inference
- **generator_only.pt**: Standalone generator model (recommended for inference)

## Configuration
See `training_config.json` for detailed training parameters and model statistics.

## Architecture
See `model_architecture.txt` for complete model architecture details.

## Usage for Inference

```python
import torch
from torchvision import transforms
from PIL import Image

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator = torch.load('generator_only.pt', map_location=device)
generator.eval()

# Prepare image
img = Image.open('sketch.jpg').convert('RGB')
img = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])(img).unsqueeze(0).to(device)

# Generate output
with torch.no_grad():
    output = generator(img)
```

## Dataset Information
- Total training samples: {}
- Image size: 256x256
- Datasets used:
  - CUHK Face-Sketch Database
  - Anime Sketch Colorization Dataset
  - Additional/Additional datasets (if available)

## Results
See `results_visualization.png` for sample outputs from the trained model.
""".format(len(dataset)))
print(f"✓ Saved README: {readme_path}")

# 9. Print final summary
print("\n" + "="*60)
print("TRAINING COMPLETE - FILES SAVED SUMMARY")
print("="*60)
print(f"Output directory: {output_dir}")
print(f"\nFiles saved:")
print(f"  • best_model.pt")
print(f"  • generator_only.pt (recommended for inference)")
print(f"  • training_config.json")
print(f"  • model_architecture.txt")
print(f"  • checkpoints/ (directory with all epoch checkpoints)")
print(f"  • README.md (usage instructions)")
if os.path.exists('/kaggle/working/results.png'):
    print(f"  • results_visualization.png")
print("\nModel Statistics:")
print(f"  • Dataset size: {len(dataset)} images")
print(f"  • Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
print(f"  • Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")
print(f"  • Total training steps: {len(dataloader) * 50:,}")
print("="*60)
print("\n✓ All files successfully saved to Kaggle output directory!")

# Optional: Create Gradio interface for inference
def process_image(image):
    generator.eval()
    with torch.no_grad():
        img = Image.fromarray(image).convert('RGB')
        img = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])(img).unsqueeze(0).to(device)
        
        output = generator(img)
        output = denormalize(output[0].cpu()).permute(1, 2, 0).numpy()
        return (output * 255).astype(np.uint8)

interface = gr.Interface(
    fn=process_image,
    inputs=gr.Image(type='numpy', label='Upload Sketch/Grayscale'),
    outputs=gr.Image(type='numpy', label='Generated Image'),
    title='Pix2Pix Image Translation - Sketch to Photo',
    description='Convert sketches to realistic images or colorize grayscale images using the trained Pix2Pix model',
    examples=None
)

interface.launch(share=True)